# RFW frozen-codec 1:1 verification

LFW 또는 SurvFace development split에서 이미 fit되어 SHA가 고정된 PCA/PQ codec을 RFW origin embeddings에 그대로 적용한다. RFW 9-fold에서 threshold를 정하고 held-out fold를 평가하며, 결과는 supplementary 1:1 verification으로만 보고한다.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun project root could not be located")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments import FrozenCodecSpec, evaluate_rfw_frozen_codecs


## 실행 설정과 frozen codec lineage

`CODEC_CONFIGS`에는 codec 파일뿐 아니라 fit dataset, run ID, fit manifest와 두 SHA를 모두 명시한다. 자동으로 최신 run을 선택하지 않는다. 현재 completed LFW/SurvFace paper run은 codec 파일을 보존하지 않았으므로, 다음 run에서 immutable codec artifact를 발행한 뒤 이 목록을 채워야 한다.


In [ ]:
MODEL_UID = "edgeface-a348c305af33c223b337"
ARTIFACT_STORAGE_MODE = "results_only"
EXECUTE_STAGE = True
WRITE_OUTPUTS = True
REUSE_COMPLETED = True
BOOTSTRAP_SEED = 42
BOOTSTRAP_REPEATS = 2000

# Example keys only. Replace with SHA-pinned artifacts from a completed fit run.
CODEC_CONFIGS = []
# CODEC_CONFIGS = [{
#     "profile_name": "pca_32",
#     "family": "pca",
#     "artifact_path": PROJECT_ROOT / "runs/.../pca_32.joblib",
#     "artifact_sha256": "<64 hex>",
#     "fit_source_dataset": "lfw",
#     "fit_source_run_id": "<completed run id>",
#     "fit_manifest_path": PROJECT_ROOT / "runs/.../manifest.json",
#     "fit_manifest_sha256": "<64 hex>",
# }]

if ARTIFACT_STORAGE_MODE not in {"results_only", "full"}:
    raise ValueError("ARTIFACT_STORAGE_MODE must be results_only or full")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True requires EXECUTE_STAGE=True")
ARTIFACT_ROOT = PROJECT_ROOT / ("results" if ARTIFACT_STORAGE_MODE == "results_only" else "runs")
ORIGIN_ARTIFACT_DIR = ARTIFACT_ROOT / "rfw_step7/origin_embeddings" / MODEL_UID
OUTPUT_DIR = ARTIFACT_ROOT / "rfw_step7/frozen_codec_evaluation" / MODEL_UID


## 평가 실행

PCA는 실제 reduced-coordinate cosine과 reconstruction cosine을 분리한다. PQ는 reconstruction cosine과 symmetric ADC-like negative squared-L2를 분리한다. codec artifact 자체의 실제 file bytes도 전체 저장량에 포함한다.


In [ ]:
codec_specs = [FrozenCodecSpec(**config) for config in CODEC_CONFIGS]
if EXECUTE_STAGE and WRITE_OUTPUTS and not codec_specs:
    raise RuntimeError(
        "CODEC_CONFIGS is empty. Publish SHA-pinned LFW/SurvFace codec artifacts first."
    )

evaluation = None
if EXECUTE_STAGE and WRITE_OUTPUTS:
    evaluation = evaluate_rfw_frozen_codecs(
        origin_artifact_dir=ORIGIN_ARTIFACT_DIR,
        codec_specs=codec_specs,
        output_dir=OUTPUT_DIR,
        strict_official=True,
        bootstrap_seed=BOOTSTRAP_SEED,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        reuse_completed=REUSE_COMPLETED,
    )
evaluation.profile_summary if evaluation else {"codec_count": len(codec_specs)}


## 해석 경계

RFW 결과에는 verification accuracy, TAR/FAR, EER와 group gap/CI만 사용한다. DIR@FPIR, open-set rank 또는 RFW에 적합한 codec이라는 표현은 사용하지 않는다. codec fit이 RFW에서 수행되지 않았음은 manifest의 `fit_on_rfw=false`로 확인한다.
